# PhysioLive

A real-time physiotherapy coach that runs on a laptop CPU. The webcam sees the exercise, the app draws the full-body skeleton with finger joints, counts each rep, and speaks per-rep feedback whenever the form deviates from the target.

## How to run

1. `pip install -r requirements.txt` (Python 3.10+).
2. In this notebook, click **Kernel > Restart & Run All**.
3. A browser tab opens at `http://localhost:8000` and shows the live camera with the skeleton overlay, the rep counter, and the coach's feedback.
4. To stop, use **Kernel > Interrupt**.

The first run downloads the pose model on demand (a one-time step).

## Step 1 - Load libraries and locate the project

Adds `src/` to Python's import path and pulls in the app modules that do the heavy lifting: the pose backend, the angle math, the rep counter, the rule engine, the voice worker, the dashboard server, and the video-source helper. OpenCV is used for camera capture and drawing.

In [ ]:
import json
import sys
import time
import webbrowser
from pathlib import Path

import cv2
import numpy as np

PROJECT_ROOT = Path().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from app.pose_gate import PoseInferencer, draw_pose, KP_MIN_CONF
from app.angles import all_angles
from app.rep_counter import RepCounter
from app.form_rules import evaluate as evaluate_rules
from app.voice import VoiceWorker
from app.dashboard_server import DashboardServer, STATE
from app.phone_stream import open_source

print(f"project root: {PROJECT_ROOT}")


## Step 2 - Pick an exercise

Each exercise lives in its own JSON file under `src/app/exercises/`. The file defines the pose backend to use, the rep goal, the state-machine thresholds that decide when a rep starts and ends, and the form rules the coach will check. Change `EXERCISE_ID` below to switch exercises without touching any code.

In [ ]:
EXERCISE_ID = "squat"

exercise_path = SRC / "app" / "exercises" / f"{EXERCISE_ID}.json"
with open(exercise_path, "r", encoding="utf-8") as f:
    EXERCISE = json.load(f)

print(f"exercise:    {EXERCISE['name']}")
print(f"backend:     {EXERCISE['pose_backend']}")
print(f"rep goal:    {EXERCISE['rep_goal']}")


## Step 3 - Load and warm the pose model

The default backend is MediaPipe Holistic, which returns 33 body landmarks plus 21 landmarks for each hand (finger articulation). A synthetic warm-up frame is passed through the model so the first real frame from the camera does not pay the initialization cost mid-loop.

In [ ]:
pose = PoseInferencer(backend=EXERCISE["pose_backend"], imgsz=384)
_ = pose.infer(np.zeros((384, 384, 3), dtype=np.uint8))
print("pose model ready.")


## Step 4 - Start the dashboard and the voice worker

The dashboard is a tiny HTTP server on `localhost:8000` that streams the annotated frames as MJPEG and serves the live session state to the browser. The voice worker runs in its own thread and speaks feedback via the system TTS engine (pyttsx3). The browser tab opens automatically.

In [ ]:
server = DashboardServer(port=8000, directory=SRC / "web")
url = server.start()
voice = VoiceWorker()
voice.start()
print(f"dashboard: {url}")
try:
    webbrowser.open(url, new=2)
except Exception:
    pass


## Step 5 - Open the video source

By default the built-in webcam (index 0) is used. Change `SOURCE` to:

- `1` for a second webcam.
- An HTTP or RTSP URL for a phone stream, e.g. `"http://192.168.1.42:8080/video"` (works with the IP Webcam / DroidCam apps).
- A path to a local video file for offline testing.

A single frame is read to confirm the source works before the main loop starts.

In [ ]:
SOURCE = 0
cap = open_source(SOURCE, width=1280, height=720, fps=30)
ok, probe = cap.read()
if not ok:
    raise RuntimeError(f"cannot read from source {SOURCE!r}")
print(f"source open: frame shape {probe.shape}")


## Step 6 - Live loop

This is the main pipeline. For every camera frame it:

1. Runs the pose backend and computes joint angles.
2. Feeds the primary angle (knee) into the rep counter's state machine.
3. When a rep closes, evaluates the form rules against the rep's angle statistics and produces a verdict.
4. Speaks the verdict through the voice worker (dedup keeps the coach quiet when the same message would repeat).
5. Draws the full skeleton (body, hands, neck) plus a heads-up display on the frame.
6. Publishes the annotated JPEG to the MJPEG stream and pushes a small JSON state to `/api/state` for the dashboard's live tab.

Interrupt the kernel to stop cleanly; the finally block releases the camera and the voice worker.

In [ ]:
rep_def = EXERCISE["rep_definition"]
rep_counter = RepCounter(
    standing_deg=rep_def["standing_deg"],
    bottom_deg=rep_def["bottom_deg"],
    hysteresis_deg=rep_def["hysteresis_deg"],
    confirm_frames=rep_def["confirm_frames"],
)

STATE.set_state({
    "running": True,
    "exercise": EXERCISE["name"],
    "rep_count": 0,
    "rep_goal": EXERCISE["rep_goal"],
    "rep_state": rep_counter.state,
    "knee_angle": None,
    "verdict": {"level": "", "text": ""},
})


def _pick_primary_side(angles):
    kl = angles.get("knee_left")
    kr = angles.get("knee_right")
    if kl is not None and kr is not None:
        return "left" if kl <= kr else "right"
    if kl is not None:
        return "left"
    if kr is not None:
        return "right"
    return None


def _draw_hud(frame, rep_count, rep_goal, knee_angle, state_text,
              verdict_text, verdict_level):
    h, w = frame.shape[:2]
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (360, 130), (16, 20, 28), -1)
    cv2.addWeighted(overlay, 0.72, frame, 0.28, 0, frame)
    cv2.putText(frame, f"Reps  {rep_count} / {rep_goal}", (24, 46),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (232, 236, 241), 2,
                cv2.LINE_AA)
    knee_txt = f"Knee  {int(knee_angle)}deg" if knee_angle is not None else "Knee  -"
    cv2.putText(frame, knee_txt, (24, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (154, 164, 178), 2,
                cv2.LINE_AA)
    cv2.putText(frame, f"State {state_text}", (24, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (154, 164, 178), 2,
                cv2.LINE_AA)
    if verdict_text:
        color = {"good": (71, 199, 106), "warn": (36, 165, 245),
                 "bad": (68, 68, 239)}.get(verdict_level, (232, 236, 241))
        cv2.rectangle(frame, (10, h - 60), (min(w - 10, 900), h - 10),
                      (16, 20, 28), -1)
        cv2.putText(frame, verdict_text[:70], (24, h - 24),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, color, 2,
                    cv2.LINE_AA)
    return frame


last_verdict_text = ""
last_verdict_level = ""
loop_t0 = time.perf_counter()
frames_seen = 0

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        result = pose.infer(frame)
        kps = result.coco17 if result else None
        angles = all_angles(kps) if kps else {}
        primary_side = _pick_primary_side(angles) if angles else None
        knee_angle = angles.get(f"knee_{primary_side}") if primary_side else None
        torso_vert = angles.get("torso_vertical") if angles else None
        torso_len = angles.get("torso_length") if angles else None
        knee_toe_norm = None
        if (primary_side
                and angles.get(f"knee_over_toe_{primary_side}") is not None
                and torso_len):
            knee_toe_norm = (angles[f"knee_over_toe_{primary_side}"]
                             / torso_len)

        event = rep_counter.update(knee_angle, torso_vertical=torso_vert,
                                   knee_over_toe_norm=knee_toe_norm)
        if event is not None:
            verdict = evaluate_rules(event.sample, EXERCISE["rules"])
            last_verdict_text = verdict.text
            last_verdict_level = verdict.level
            voice.say(last_verdict_text)

        if result is not None:
            frame = draw_pose(frame, result, min_conf=KP_MIN_CONF)
        frame = _draw_hud(frame, rep_counter.count, EXERCISE["rep_goal"],
                          knee_angle, rep_counter.state,
                          last_verdict_text, last_verdict_level)

        STATE.set_state({
            "running": True,
            "exercise": EXERCISE["name"],
            "rep_count": rep_counter.count,
            "rep_goal": EXERCISE["rep_goal"],
            "rep_state": rep_counter.state,
            "knee_angle": knee_angle,
            "verdict": {"level": last_verdict_level,
                        "text": last_verdict_text},
        })

        ok_enc, jpeg = cv2.imencode(".jpg", frame,
                                    [int(cv2.IMWRITE_JPEG_QUALITY), 72])
        if ok_enc:
            STATE.push_frame(jpeg.tobytes())

        frames_seen += 1
        if frames_seen % 60 == 0:
            fps = frames_seen / max(1e-6, time.perf_counter() - loop_t0)
            print(f"~{fps:.1f} fps, reps: {rep_counter.count}")
except KeyboardInterrupt:
    print("interrupted by user.")
finally:
    cap.release()
    voice.stop()
    STATE.set_state({**STATE.get_state(), "running": False})
    print(f"final rep count: {rep_counter.count}")
